Encoder Only

In [2]:
import pandas as pd
from sentence_transformers import SentenceTransformer

# Read the CSV file
df = pd.read_csv('../large_idiom_set_fr_eng.csv')


df['french_text'] = df['original_text']
df['english_text'] = df['text']

In [18]:
# Load multilingual embedding model (works well for English and French)
model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')

# Generate embeddings for English texts
print("Generating English embeddings...")
english_embeddings = model.encode(df['english_text'].tolist(), show_progress_bar=True)

# Generate embeddings for French texts
print("Generating French embeddings...")
french_embeddings = model.encode(df['french_text'].tolist(), show_progress_bar=True)

# Create separate DataFrames with embeddings
english_df = pd.DataFrame({
    'text': df['english_text'],
    'language': 'English',
    'embedding': list(english_embeddings)
})

french_df = pd.DataFrame({
    'text': df['french_text'],
    'language': 'French',
    'embedding': list(french_embeddings)
})

# Combine both DataFrames
combined_df = pd.concat([english_df, french_df], ignore_index=True)

Generating English embeddings...


Batches:   0%|          | 0/207 [00:00<?, ?it/s]

Generating French embeddings...


Batches:   0%|          | 0/207 [00:00<?, ?it/s]

In [19]:
# Reduce embeddings to 2D using UMAP
from umap import UMAP
import numpy as np

print("Reducing embeddings to 2D...")
all_embeddings = np.vstack(combined_df['embedding'].values)
reducer = UMAP(n_components=2, random_state=42)
embeddings_2d = reducer.fit_transform(all_embeddings)

# Add 2D coordinates to dataframe
combined_df['x'] = embeddings_2d[:, 0]
combined_df['y'] = embeddings_2d[:, 1]

# Save to CSV for online visualization
output_df = combined_df[['text', 'language', 'x', 'y']].copy()
output_df.to_csv('embeddings_2d.csv', index=False)
print("Saved embeddings to 'embeddings_2d.csv'")

# Also save with full embeddings in parquet format
combined_df.to_parquet('embeddings_full.parquet')
print("Saved full embeddings to 'embeddings_full.parquet'")

print(f"\nDataset info: {len(output_df)} points total")
print(f"English: {len(output_df[output_df['language']=='English'])} points")
print(f"French: {len(output_df[output_df['language']=='French'])} points")

Reducing embeddings to 2D...


C:\Users\benja\AppData\Roaming\Python\Python312\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Saved embeddings to 'embeddings_2d.csv'
Saved full embeddings to 'embeddings_full.parquet'

Dataset info: 13232 points total
English: 6616 points
French: 6616 points


In [20]:
import numpy as np

# Get x, y coordinates for English and French
english_points = embeddings_2d[:len(english_embeddings)]
french_points = embeddings_2d[len(english_embeddings):]

# Calculate Euclidean distance between corresponding points
distances = []
for i in range(len(english_points)):
    dist = np.sqrt((english_points[i][0] - french_points[i][0])**2 + 
                   (english_points[i][1] - french_points[i][1])**2)
    distances.append(dist)

# Compute average distance
avg_distance = np.mean(distances)

print(f"Average distance between English and French: {avg_distance:.4f}")
print(f"Min distance: {np.min(distances):.4f}")
print(f"Max distance: {np.max(distances):.4f}")

Average distance between English and French: 2.2786
Min distance: 0.0006
Max distance: 14.4796


In Terminal : embedding-atlas Code1MiniLML12/embeddings_full.parquet --x x --y y --text text